# Assignment 2024 - 2025: NLP Sentiment Analysis

# Team: Vasilis Kalfopoulos(Email:vkalfopoulos@ihu.edu.gr) , Antonis Koutsoupias (Email:akoutsoupias@ihu.edu.gr)

## 1. Introduction

Natural Language Processing (NLP) is presently among the hottest scientific fields with an enormous growth rate of the relevant research. Sentiment analysis is a popular NLP problem that aims at the automatic identification of the polarity in user reviews, tweets, blog posts, comments, forum discussions and so on.

In the vast majority of cases, sentiment analysis is treated as a text classification problem. If the involved text polarity is binary (i.e., positive or
negative, good or bad), then binary text classification models are trained by
utilizing two class labels. On the other hand, in case the polarity falls into a
closed score range (e.g., 1–5, 1–10, etc.), then each individual score is treated
as a separate class label and multi-class classification approaches are applied.

In this assignment you will explore various classifiers on NLP sentiment analysis. The purpose is to measure their performance on a dataset that derives from X (formerly Twitter) and contains user opinions about a US Airliner.


### Classifiers
The classifiers to be studied are:
* $C_1$: Logistic Regression
* $C_2$: Support Vector Machines (Linear kernel) - hint: use [`LinearSVC`](https://scikit-learn.org/dev/modules/generated/sklearn.svm.LinearSVC.html)
* $C_3$: Random Forests
* $C_4$: Feed-forward Neural Network


### Dataset

The classification performance of the abovementioned models will be studied on the `Twitter_US_Airline_Sentiment.csv` dataset ([see more details here](https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment)).

**The dataset is provided with the assignment in the present compressed file.**



## 2. Experiments

You will vectorize the text (located in the column `text`) by using the well-known TF-IDF technique. There will be three cases where the vocabulary of `TfidfVectorizer` will be limited to:

1. Contain words that appear in at least 5 documents (hint: `min_df` parameter of `TfidfVectorizer`).
2. Contain 2500 words (hint: `max_features` parameter of `TfidfVectorizer`).
3. Contain 500 words (hint: `max_features` parameter of `TfidfVectorizer`).

The classifiers will be evaluated by using 5-fold cross validation. Make sure that no information will be leaked from the training set to the test set. The values of the four following metrics will be measured:

* $M_1$: Accuracy
* $M_2$: F1-score
* $M_3$: Fit time


## 3. Deliverable & Deadline

You must work in teams of two. The deliverable must be **this** notebook, **renamed using both your surnames in alphabetical order as the final file name**. Both students of a team must upload the same file to the e-learning platform. **Only one file per team will be checked. So, if you upload different versions, then only one of them will not be examined and evaluated**.  Your notebook must include the code and the results for each experiment. You must also provide a brief discussion on the performance of the classifiers and the leasons learned.

<div style="border:1px solid black; font-weight:bold; width:100%; text-align:center; height:50px; line-height:50px; font-size:12pt">The deadline is 05/02/2024, 22:00 hrs. NO DEADLINE EXTENSION WILL BE GIVEN.</div>

## Solution

Please write your solution here, including your code and descriptions. **Do not modify the notebook's structure**.


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import make_scorer, accuracy_score, f1_score

# Dataset location 
DATASET_LOCATION = r"C:\Users\Antonis\Documents\University\DataScience\ML\Twitter_US_Airline_Sentiment.csv"

# Load dataset
data = pd.read_csv(DATASET_LOCATION, sep=',')

# OR
# data = pd.read_csv("Twitter_US_Airline_Sentiment.csv")

# Examine dataset structure
print("Dataset Preview:\n", data.head()) # Display the first 5 rows of the dataset
print("\nDataset Columns:", data.columns) # Display the names of all columns in the dataset

# OR
# print(data.head())

# Extract relevant columns
X = data['text'].fillna("")  # Handling missing values by replacing NaN with empty string
y = data['airline_sentiment'] # The target labels (positive, neutral, negative)

# Function to preprocess text using TF-IDF
def preprocess_tfidf(X, min_df=1, max_features=None):
    vectorizer = TfidfVectorizer(min_df=min_df, max_features=max_features)
    X_tfidf = vectorizer.fit_transform(X)
    print(f"TF-IDF matrix shape: {X_tfidf.shape}")  # Debugging
    return X_tfidf, vectorizer

# Classifiers dictionary
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Support Vector Machines": LinearSVC(dual=False, max_iter=5000),  # Avoids convergence issues
    "Random Forests": RandomForestClassifier(),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(50,), max_iter=500)  # Smaller network for speed
}

# Function to evaluate classifiers using K-fold cross-validation
def evaluate_classifiers(X, y, classifiers):
    results = {}
    # Define performance metrics to be used in evaluation
    metrics = {
        'accuracy': make_scorer(accuracy_score), # Prediction accuracy metric
        'f1_score': make_scorer(f1_score, average='weighted'), # Metric to account for imbalanced data
    }
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) # Folds creation for cross-validation
    # Iterate over each classifier in the dictionary and evaluate its performance
    for name, clf in classifiers.items():
        try:
            print(f"Evaluating {name}...")  # Debugging
            scores = cross_validate(clf, X, y, cv=skf, scoring=metrics, return_train_score=False)
            results[name] = {
                'Accuracy': np.mean(scores['test_accuracy']),
                'F1-score': np.mean(scores['test_f1_score']),
                'Fit time': np.mean(scores['fit_time'])
            }
        except Exception as e:
            print(f"Error evaluating {name}: {e}")  # Prevents crashes and continues
            results[name] = {'Error': str(e)}
    return results

# Define cases with parameters for TF-IDF
cases = {
    "Case 1 (min_df=5)": {"min_df": 5},
    "Case 2 (max_features=2500)": {"max_features": 2500},
    "Case 3 (max_features=500)": {"max_features": 500}
}

# Dictionary to store results
results = {}

# Running different cases
for case_name, params in cases.items():
    print(f"\nRunning {case_name}...")
    X_tfidf, _ = preprocess_tfidf(X, **params)  # Preprocess data
    results[case_name] = evaluate_classifiers(X_tfidf, y, classifiers)
    print(f"{case_name} evaluation completed !")

# Print final results
for case, res in results.items():
    print(f"\n{case}")
    for clf_name, metrics in res.items():
        print(f"{clf_name}: {metrics}")


Dataset Preview:
              tweet_id airline_sentiment  airline_sentiment_confidence  \
0  570306133677760513           neutral                        1.0000   
1  570301130888122368          positive                        0.3486   
2  570301083672813571           neutral                        0.6837   
3  570301031407624196          negative                        1.0000   
4  570300817074462722          negative                        1.0000   

  negativereason  negativereason_confidence         airline  \
0            NaN                        NaN  Virgin America   
1            NaN                     0.0000  Virgin America   
2            NaN                        NaN  Virgin America   
3     Bad Flight                     0.7033  Virgin America   
4     Can't Tell                     1.0000  Virgin America   

  airline_sentiment_gold        name negativereason_gold  retweet_count  \
0                    NaN     cairdin                 NaN              0   
1             

# Summary
After evaluating 4 different classifiers (Logistic Regression, Support Vector Machines, Random Forests, Neural Networks) using accuracy and f1-score as performance metrics, we can deduct the following:
- **Logistic Regression** works really well with linearly seperable data, although it lacks in performance with more complex data.
- **Support Vector Machines (SVM)** perform reasonably well with high-dimensional data.
- **Random Forests** perform quite well with noisy data, but the training time is higher that the previous 2 classifier.
- **Neural Networks** can be tuned to perform really well with complex data, but that requires more time and computational resources than the other classifiers.

From the performance metrics, we come to the conclusion that Logistic Regression and Support Vector Machines were better at predicting the sentiment of the tweets while also having better fit time, so the most complex classifier is not always the best one.
